# Lab grading console (instructor)

Buttons for the weekly grading loop, so you never touch the terminal. Each button runs the
matching script and **streams its output live** into the log below, so you can watch every
student/question as it is graded and see exactly what happens under the hood.

**Launch it** from the `lab01/` folder so the `ollama` package is on the path:
```bash
cd lab01
uv run jupyter notebook _solutions/grading/grading_console.ipynb
```
Then run the single code cell below (Shift+Enter) to get the buttons.

**The four steps:** 1 collect+grade -> 2 preview (safe) -> 3 publish to students -> 4 read appeals.
Steps 1, 3, 4 read/write student home dirs, so they use `sudo`. If a button prints
*"sudo: a password is required"*, add a NOPASSWD rule for these scripts (see the grading
`README.md`) or launch this notebook from a session where `sudo` is already authorized.
Step 2 (preview) and grading itself need no sudo.

In [ ]:
import os, sys, re, json, subprocess
import ipywidgets as w
from IPython.display import display

# Paths are derived from where this notebook lives (lab01/_solutions/grading/).
LAB = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))      # -> lab01/
G = os.path.join('_solutions', 'grading')                         # scripts, relative to LAB
GABS = os.path.join(LAB, G)
PY = sys.executable                                               # the uv env python (has `ollama`)
STATUS_FILE = os.path.join(LAB, 'submissions', 'status.json')     # written by the scripts
# Demo mode: grade three dummy students (demo/homes) instead of real ones. No sudo, no risk.
DEMO_HOMES = os.path.join(GABS, 'demo', 'homes', 'jupyter-*')
DEMO_SUB = os.path.join(GABS, 'demo', 'submissions')
DEMO_STATUS = os.path.join(DEMO_SUB, 'status.json')
PROG = re.compile(r'@PROGRESS (\d+)/(\d+)')

CSS = """<style>
.gconsole .widget-button{border-radius:8px !important;font-weight:600}
.gpanel{border:1px solid #8883;border-radius:10px;padding:10px 14px;margin:6px 0;
  background:#8881;font-size:13px}
.gpanel b{font-size:13px}
.glog{background:#0f141b;border:1px solid #223 !important;border-radius:10px !important}
.glog pre{font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:12px;
  color:#cdd9e5;line-height:1.45}
</style>"""

def local_models():
    """Model names installed in the local Ollama (for the judge dropdown)."""
    try:
        out = subprocess.run(['ollama', 'list'], capture_output=True, text=True, timeout=10).stdout
        return [ln.split()[0] for ln in out.splitlines()[1:] if ln.strip()]
    except Exception:
        return []

lab_box = w.Text(value=os.environ.get('LAB', 'lab01'), description='LAB:',
                 layout=w.Layout(width='190px'))
judge_box = w.Dropdown(options=['(default: nemotron-3-super:latest)'] + local_models(),
                       value='(default: nemotron-3-super:latest)', description='JUDGE_MODEL:',
                       style={'description_width': 'initial'}, layout=w.Layout(width='430px'))
demo_chk = w.Checkbox(value=False, description='Demo mode (3 dummy students, no sudo)',
                      indent=False, layout=w.Layout(width='340px'))
panel = w.HTML()
bar = w.IntProgress(value=0, min=0, max=1, description='idle',
                    style={'description_width': '80px'}, layout=w.Layout(width='430px'))
status = w.HTML()
log = w.Output(layout=w.Layout(padding='8px 12px', height='320px', overflow='auto'))
log.add_class('glog')
panel.add_class('gpanel')

def refresh_panel(*_):
    """Show when each step last ran (and with which model), so you never double-grade."""
    try:
        st = json.load(open(DEMO_STATUS if demo_chk.value else STATUS_FILE, encoding='utf-8'))
    except Exception:
        st = {}
    def line(key, label, extra):
        d = st.get(key)
        if not d:
            return f"<tr><td style='padding:2px 14px 2px 0;color:#888'>{label}</td><td style='color:#c0392b'>never</td></tr>"
        return (f"<tr><td style='padding:2px 14px 2px 0;color:#888'>{label}</td>"
                f"<td><b>{d['time']}</b> &nbsp;<span style='color:#888'>{extra(d)}</span></td></tr>")
    rows = (line('graded', 'last collect + grade',
                 lambda d: f"{d.get('model','?')} &middot; {d.get('students',0)} students, {d.get('graded',0)} notebooks")
            + line('published', 'last publish', lambda d: f"{d.get('published',0)} students")
            + line('appeals', 'last read appeals', lambda d: f"{d.get('count',0)} appeal(s)"))
    tag = " <span style='color:#c0392b'>(DEMO)</span>" if demo_chk.value else ""
    panel.value = f"<b>Status</b>{tag}<table style='margin-top:6px'>{rows}</table>"

demo_chk.observe(refresh_panel, 'value')

def run(script, sudo=False, extra=None):
    """Run a grading script from lab01/, streaming output live into the log.

    Uses log.append_stdout + `log.outputs = ()` to reset (NOT `with log:` / clear_output(),
    which duplicate or fail to clear). LAB / JUDGE_MODEL (and, in demo mode, HOMES/OUT) are
    injected via an `env VAR=val` prefix so they survive `sudo` (sudo strips the environment).
    Demo mode targets the dummy students and never uses sudo.
    """
    for b in buttons:
        b.disabled = True
    log.outputs = ()
    status.value = "<b style='color:#b36b00'>running&hellip;</b>"
    bar.value, bar.max, bar.bar_style, bar.description = 0, 1, 'info', 'working'
    demo = demo_chk.value
    envargs = [f"LAB={lab_box.value.strip() or 'lab01'}"]
    if demo:
        envargs += [f"HOMES={DEMO_HOMES}", f"OUT={DEMO_SUB}", f"SUBMISSIONS={DEMO_SUB}"]
    if not judge_box.value.startswith('(default'):     # else grade.py uses its own default
        envargs.append(f'JUDGE_MODEL={judge_box.value}')
    cmd = (['sudo', '-n'] if (sudo and not demo) else []) + ['env'] + envargs + \
          [PY, os.path.join(G, script)] + (extra or [])
    log.append_stdout('$ ' + ' '.join(cmd) + '\n\n')
    rc, saw_progress = -1, False
    try:
        p = subprocess.Popen(cmd, cwd=LAB, text=True, bufsize=1,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        for ln in p.stdout:
            m = PROG.match(ln.strip())
            if m:                                       # progress marker -> drive the bar, don't print
                done, total = int(m.group(1)), int(m.group(2))
                bar.max = max(total, 1)
                bar.value = done
                bar.description = f'{done} / {total}'
                saw_progress = True
                continue
            log.append_stdout(ln)
        rc = p.wait()
    except Exception as exc:
        log.append_stdout(f'! could not run: {exc}\n')
    log.append_stdout('\n' + '-' * 60 + f'  [exit {rc}]\n')
    ok = rc == 0
    status.value = ("<b style='color:#0a0'>done &check;</b>" if ok
                    else f"<b style='color:#c00'>exit {rc}</b>")
    bar.bar_style = 'success' if ok else 'danger'
    if ok and not saw_progress:
        bar.max, bar.value, bar.description = 1, 1, 'done'
    refresh_panel()
    for b in buttons:
        b.disabled = False

# Each button just calls the matching script. Steps that touch student homes use sudo.
def collect_grade(_): run('collect_and_grade.py', sudo=True)
def preview(_):       run('publish_grades.py', extra=['--dry-run'])
def gradebook(_):     run('build_gradebook.py')
def publish(_):       run('publish_grades.py', sudo=True)
def appeals(_):       run('read_appeals.py', sudo=True)
def clear(_):         log.outputs = ()

def mk(desc, style, icon):
    return w.Button(description=desc, button_style=style, icon=icon, layout=w.Layout(width='198px'))
b1 = mk('1 - Collect + grade', 'primary', 'download')
b2 = mk('2 - Preview (safe)', '', 'eye')
bG = mk('View gradebook', 'success', 'table')
b3 = mk('3 - Publish to students', 'warning', 'paper-plane')
b4 = mk('4 - Read appeals', '', 'inbox')
bc = w.Button(description='clear log', icon='trash', layout=w.Layout(width='120px'))
buttons = [b1, b2, bG, b3, b4, bc]
b1.on_click(collect_grade); b2.on_click(preview); bG.on_click(gradebook)
b3.on_click(publish); b4.on_click(appeals); bc.on_click(clear)

def grp(title, *widgets):
    return w.VBox([w.HTML(f"<span style='color:#888;font-size:12px'>{title}</span>"),
                   w.HBox(list(widgets))])

refresh_panel()
ui = w.VBox([
    w.HTML(CSS + '<h3 style="margin:2px 0">Lab grading console</h3>'
           '<span style="color:#888">weekly loop: collect+grade &rarr; preview &rarr; '
           'gradebook &rarr; publish &rarr; appeals</span>'),
    panel,
    w.HBox([lab_box, judge_box]),
    demo_chk,
    grp('grade &amp; review', b1, b2, bG),
    grp('release', b3, b4),
    w.HBox([bar, status]),
    w.HBox([bc]),
    log,
])
ui.add_class('gconsole')
display(ui)